# Synthetic Market Data Generator (Group 00 · Step 00)

> **What this notebook does** — generates a **synthetic** market-data bucket on Kaggle: 20,000 fake tickers of minute-bar OHLCV data with realistic sector/lead-lag structure. It fakes **only the raw layer** (what Massive / Yahoo would provide); everything else (daily summaries, correlations, backtests) is derived by the existing pipeline (`02-01` → `03-01` → `04-01`) running **unchanged** against the published dataset.

## Why synthetic?

- The real market data is licensed (Massive/Polygon terms) and cannot be republished. All data here is **synthetically generated** (CC0) — no real prices, no copyright issues.
- The dataset is **public** (`dsptlp/synthetic-market-data`) so it doesn't count against private quota and can be used in public example notebooks.

## How it builds a 20,000-ticker dataset inside Kaggle's 9-hour limit

One run cannot generate + upload everything, so this notebook is **resumable**:

1. **Mount** the previous dataset version (bootstrap on run #1).
2. **Copy** it into the staging area (hidden Kaggle disk space, up to ~200 GB).
3. **Generate** tickers one at a time in a loop until the time budget is hit (`MAX_GENERATION_MINUTES`, 30 min in test mode).
4. **Publish** a new dataset version.
5. **Re-run** (re-push the kernel) to keep adding tickers until `TOTAL_TICKERS` is reached.

Resume is crash-safe: each ticker is one parquet file and generation is deterministic (`SEED + ticker_index`), so a ticker is never generated twice and a killed run only loses the in-flight ticker.

## Output (mirrors the real S3 bucket layout)

| Path in dataset | Content |
|---|---|
| `parquet_data/minute_data_final/{TICKER}.parquet` | one file per ticker, minute bars |
| `parquet_data/summary/tickers/tickers.parquet` | ticker metadata (`ticker`, `type`, ...) |
| `parquet_data/types/ticker_types.parquet` | asset-class lookup table |
| `parquet_data/manifest.json` | resume state (done tickers, params, run log) |


## Run instructions

- **Run #1**: pushes a new public dataset version (bootstrap).
- **Runs #2+**: mounts the previous version, appends the next batch, publishes again.
- Repeat until the summary says `20000/20000`. Each run is a manual re-push of this kernel.
- Tune `MAX_GENERATION_MINUTES` for longer test runs once happy with the 30-minute cycles.


In [ ]:

# ============================================================================
# ALL PARAMETERS (single source of truth)
# ============================================================================

# --- Dataset ---------------------------------------------------------------
# Shard 1 (dataset synthetic-market-data) already holds SGT00000..SGT11229.
# This run builds SHARD 2 (SGT11230..SGT19999) into its own dataset, so every
# published version stays well under the sizes that have proven reliable.
DATASET_SLUG    = "synthetic-market-data-02"
DATASET_TITLE   = "Synthetic Market Data (SGT Universe) - Shard 2"
DATASET_PRIVATE = False           # public -> does not count toward quota

# --- Shard ticker range (global indices; per-ticker seed = SEED + index) ---
TICKER_START    = 11230
TICKER_END      = 20000

# --- Universe --------------------------------------------------------------
TOTAL_TICKERS     = 20000         # total synthetic tickers to build across runs
SEED              = 42            # master seed (deterministic, resume-safe)
N_SECTORS         = 200           # common-factor groups (100 tickers each)
N_FAKE_ETFS       = 100           # type=ETF tickers (demos the 03-01 exclusion filter)
FRACTION_DEFECTIVE = 0.005        # share with random gaps (demos quality filters)

# --- Date window (matches 03-01 correlation window) ------------------------
START_DATE = "2024-02-01"
END_DATE   = "2026-01-01"

# --- Intraday --------------------------------------------------------------
BARS_PER_DAY = 390                # full minute bars (9:30-16:00 ET)
MINUTE_STEP  = 1                  # minutes per bar

# --- Time budgets (Kaggle session cap ~9h) ---------------------------------
MAX_GENERATION_MINUTES = 30       # generate tickers for this long, then publish
MAX_RUNTIME_MINUTES   = 480       # hard stop; publish must start with margin
SAFETY_MINUTES        = 60        # reserved for copy + upload
MAX_LOCAL_GB          = 180       # stop generating if staging disk fills

# --- Price / volume realism -------------------------------------------------
START_PRICE_LO  = 5.0
START_PRICE_HI  = 250.0
GLOBAL_DAILY_VOL = 0.005          # market-wide factor daily vol
SECTOR_DAILY_VOL = 0.012          # sector factor daily vol
IDIO_DAILY_VOL  = 0.008           # idiosyncratic daily vol
MIN_AVG_VOLUME  = 20000           # well above the pipeline's 1000 liquidity floor
MAX_AVG_VOLUME  = 2000000
VOL_RESPONSE    = 5.0             # volume multiplier response to |daily return|
TRADES_PER_VOL  = 100.0           # ~1 trade per N shares
BETA_LEAD       = 1.5             # follower's loading on its sector leader (lagged)
FOLLOWER_GLOBAL_BETA = 0.05       # follower's loading on the market-wide factor
FOLLOWER_IDIO_VOL = 0.003         # follower idiosyncratic daily vol (keeps lead-lag corr high)
LEADER_IDIO_VOL   = 0.004         # leader idiosyncratic daily vol
SECTOR_AR_RHO     = 0.6           # sector-factor AR(1) persistence


In [ ]:

# ============================================================================
# Setup: imports, Kaggle auth, disk-space probe (hidden space up to ~200 GB)
# ============================================================================
import os, sys, json, time, math, glob, shutil
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# --- Kaggle auth from env vars (Add-ons -> Secrets on Kaggle) ------------
KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "dsptlp")
KAGGLE_API_TOKEN = os.environ.get("KAGGLE_API_TOKEN", "")

os.makedirs("/root/.kaggle", exist_ok=True)
if KAGGLE_API_TOKEN:
    with open("/root/.kaggle/access_token", "w") as f:
        f.write(KAGGLE_API_TOKEN)
    os.chmod("/root/.kaggle/access_token", 0o600)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME

# --- Pick the largest writable disk (hidden Kaggle space up to ~200 GB) ----
def largest_free_path(candidates):
    best, best_gb = None, -1.0
    for c in candidates:
        try:
            os.makedirs(c, exist_ok=True)
            gb = shutil.disk_usage(c).free / 1e9
            if gb > best_gb:
                best, best_gb = c, gb
        except Exception:
            pass
    return best, best_gb

WORK_ROOT, FREE_GB = largest_free_path(["/kaggle/working", "/kaggle", "/tmp", "/root"])
UPLOAD_DIR = os.path.join(WORK_ROOT, "upload")
os.makedirs(UPLOAD_DIR, exist_ok=True)

# Upgrade the bundled kaggle CLI (Kaggle image ships 2.0.2 whose dataset-upload
# API is rejected with 403 on the current platform).
!pip install -q --upgrade kaggle
!kaggle --version

print(f"Work root: {WORK_ROOT}  (free {FREE_GB:.1f} GB)")


In [ ]:

# ============================================================================
# Load previous dataset version (resume) / bootstrap on first run
# ============================================================================
# Locate the mounted previous version wherever Kaggle puts it.
# Datasets may mount at /kaggle/input/<slug> OR /kaggle/input/datasets/<owner>/<slug>,
# and the zip upload's contents land at the dataset ROOT (manifest.json + dirs).
import glob as _glob
print("input dirs:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "none")
MOUNT_DATA = None
for _cand in _glob.glob(f"/kaggle/input/**/{DATASET_SLUG}/manifest.json", recursive=True):
    MOUNT_DATA = os.path.dirname(_cand)
    break
if MOUNT_DATA is None:                       # fallback: any manifest.json under input
    for _cand in _glob.glob("/kaggle/input/**/manifest.json", recursive=True):
        MOUNT_DATA = os.path.dirname(_cand)
        break
print("mount data dir:", MOUNT_DATA)
LOCAL_DATA = os.path.join(UPLOAD_DIR, "parquet_data")

run_number = 1
manifest = {"run_log": []}

if MOUNT_DATA and os.path.isdir(MOUNT_DATA):
    # copy the previous version into the staging area (fast local copy)
    os.makedirs(LOCAL_DATA, exist_ok=True)
    for entry in sorted(os.listdir(MOUNT_DATA)):
        src = os.path.join(MOUNT_DATA, entry)
        dst = os.path.join(LOCAL_DATA, entry)
        if os.path.isdir(src) and not os.path.exists(dst):
            print(f"  copying {entry}/ ...")
            shutil.copytree(src, dst)
        elif os.path.isfile(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
    mf = os.path.join(MOUNT_DATA, "manifest.json")
    if os.path.isfile(mf):
        with open(mf) as f:
            manifest = json.load(f)
        run_number = len(manifest.get("run_log", [])) + 1
    print(f"Resumed from previous dataset version -> run #{run_number}")
else:
    print("No previous version found -> bootstrap run #1")

# Source of truth for "already done": ticker parquet files on disk
done_dir = os.path.join(LOCAL_DATA, "minute_data_final")
os.makedirs(done_dir, exist_ok=True)
done = set()
for f in glob.glob(os.path.join(done_dir, "*.parquet")):
    done.add(os.path.splitext(os.path.basename(f))[0])
print(f"Tickers already generated: {len(done):,} / {TOTAL_TICKERS:,}")


In [ ]:

# ============================================================================
# Deterministic universe + shared market/sector factor time series
# ============================================================================
master_rng = np.random.default_rng(SEED)

symbols   = [f"SGT{i:05d}" for i in range(TOTAL_TICKERS)]
sector_of = np.arange(TOTAL_TICKERS) % N_SECTORS
lag_of    = (2 + sector_of % 9).astype(int)      # sector lead-lag 2..10 days
is_leader = np.arange(TOTAL_TICKERS) < N_SECTORS # index 0..N_SECTORS-1 lead their sector
ticker_types = np.where(np.arange(TOTAL_TICKERS) < N_FAKE_ETFS, "ETF", "CS")

trading_days = np.array(pd.bdate_range(START_DATE, END_DATE).date, dtype="datetime64[D]")
n_days = len(trading_days)
print(f"{n_days:,} trading days ({START_DATE} -> {END_DATE})")

global_factor = master_rng.normal(0.0, GLOBAL_DAILY_VOL, n_days)

# Sector factors: AR(1) persistent process (realistic regimes + common trends)
_eps = master_rng.normal(0.0, SECTOR_DAILY_VOL * math.sqrt(1 - SECTOR_AR_RHO**2),
                         (N_SECTORS, n_days))
sector_factor = np.zeros((N_SECTORS, n_days))
sector_factor[:, 0] = _eps[:, 0]
for t in range(1, n_days):
    sector_factor[:, t] = SECTOR_AR_RHO * sector_factor[:, t - 1] + _eps[:, t]

# Per-leader idiosyncratic component (deterministic, shared with followers)
leader_idio = master_rng.normal(0.0, LEADER_IDIO_VOL, (N_SECTORS, n_days))

# Precomputed daily return series of each sector's leader ticker (index == sector).
# Followers are built as BETA_LEAD * leader_ret[sector, t-lag] + noise, so the
# lead-lag search (corr(leader[t], follower[t+lag])) finds genuine signals.
leader_ret = (FOLLOWER_GLOBAL_BETA * global_factor[None, :]
              + 0.3 * sector_factor
              + leader_idio)

start_prices  = master_rng.uniform(START_PRICE_LO, START_PRICE_HI, TOTAL_TICKERS)
base_volumes  = np.exp(master_rng.uniform(np.log(MIN_AVG_VOLUME), np.log(MAX_AVG_VOLUME), TOTAL_TICKERS))

# Minute-of-day offsets (09:30 -> close ET), converted to epoch-ms instants.
# Real pipeline stores ET wall-clock times as UTC instants; use US/Eastern tz.
import zoneinfo
TZ_ET = zoneinfo.ZoneInfo("US/Eastern")
MINS = (np.arange(BARS_PER_DAY) * MINUTE_STEP + 9 * 60 + 30).astype(np.int64)
_day_ns = trading_days.astype("datetime64[ns]").astype("int64")
_wall = (_day_ns[:, None] + MINS.astype(np.int64)[None, :] * 60_000_000_000)
_wall = _wall.ravel()                                          # naive ET wall clock (ns)
_bar_dt = (pd.to_datetime(_wall, unit="ns")
           .tz_localize(TZ_ET).tz_convert("UTC"))
bar_date_ms = (_bar_dt.astype("int64").to_numpy() // 10**6).ravel()   # epoch ms
n_rows_per_ticker = n_days * BARS_PER_DAY


In [ ]:

# ============================================================================
# Generation loop: one ticker at a time until the time budget is exhausted
# ============================================================================
run_start  = time.time()
gen_deadline   = run_start + MAX_GENERATION_MINUTES * 60
hard_deadline  = run_start + (MAX_RUNTIME_MINUTES - SAFETY_MINUTES) * 60

def gen_ticker(i):
    """Generate minute bars for ticker i. Deterministic: seed = SEED + i."""
    rng = np.random.default_rng(SEED + i)
    sector, lag, leader = sector_of[i], lag_of[i], is_leader[i]

    # --- daily log-returns -------------------------------------------------
    if leader:
        daily = leader_ret[sector].copy()                # matches precomputed series
    else:
        # follower reacts to its sector leader with `lag` days delay
        lead = leader_ret[sector]
        shifted = np.roll(lead, lag)
        shifted[:lag] = lead[:lag]                       # no look-ahead at start
        daily = (BETA_LEAD * shifted
                 + FOLLOWER_GLOBAL_BETA * global_factor
                 + rng.normal(0.0, FOLLOWER_IDIO_VOL, n_days))

    # --- deliberate defects (random gaps) for quality-filter demos ---------
    keep = np.ones(n_days, dtype=bool)
    if i >= TOTAL_TICKERS * (1 - FRACTION_DEFECTIVE):
        drop_frac = rng.uniform(0.05, 0.20)
        keep = rng.random(n_days) > drop_frac
        if keep.sum() < 20:                      # keep it usable
            keep = np.ones(n_days, dtype=bool)
    nd = int(keep.sum())

    # --- log-price path ------------------------------------------------------
    # Day t runs from the previous close to close[t] = close[t-1]*exp(daily[t]),
    # so the realized close-to-close return is exactly `daily` (keeps the
    # lead-lag correlation structure intact).
    close_log = np.cumsum(daily)
    close_log = close_log - close_log[0] + math.log(start_prices[i])
    prev_close_log = np.empty_like(close_log)
    prev_close_log[0] = close_log[0] - daily[0]
    prev_close_log[1:] = close_log[:-1]
    open_log_d = prev_close_log + rng.normal(0.0, IDIO_DAILY_VOL * 0.1, n_days)

    grid = np.linspace(0.0, 1.0, BARS_PER_DAY + 1)     # BARS+1 points -> BARS bars
    sig = IDIO_DAILY_VOL * 0.25 / math.sqrt(BARS_PER_DAY)
    bridge = np.cumsum(rng.normal(0.0, sig, (n_days, BARS_PER_DAY + 1)), axis=1)
    bridge = bridge - bridge[:, [-1]] * grid[None, :]   # zero at both ends
    path_log = (open_log_d[:, None]
                + (close_log[:, None] - open_log_d[:, None]) * grid[None, :]
                + bridge)
    path = np.exp(path_log)                              # (n_days, BARS+1)

    # per-minute bar OHLC from adjacent path points (last bar close = daily close)
    o = path[:, :-1]
    c = path[:, 1:]
    hi = np.maximum(o, c) * (1 + np.abs(rng.normal(0, 0.0004, o.shape)))
    lo = np.minimum(o, c) * (1 - np.abs(rng.normal(0, 0.0004, o.shape)))

    # --- volume / trades ---------------------------------------------------
    daily_vol = (base_volumes[i] * (1 + VOL_RESPONSE * np.abs(daily))
                 * np.exp(rng.normal(0, 0.3, n_days)))
    vol = (daily_vol[:, None] / BARS_PER_DAY) * np.exp(rng.normal(0, 0.5, o.shape))
    trades = np.maximum(1, np.round(vol / TRADES_PER_VOL)).astype(np.int64)
    vwap = (o + hi + lo + c) / 4.0

    # --- keep only non-defective days, flatten -----------------------------
    keep_idx = np.flatnonzero(keep)
    rows = keep_idx.size * BARS_PER_DAY
    mask = np.zeros(n_days * BARS_PER_DAY, dtype=bool)
    for k in keep_idx:
        mask[k * BARS_PER_DAY:(k + 1) * BARS_PER_DAY] = True

    tbl = pa.table({
        "symbol": pa.array([symbols[i]] * rows, type=pa.string()),
        "date":   pa.array(bar_date_ms[mask], type=pa.int64()),
        "open":   pa.array(np.round(o.ravel()[mask], 2), type=pa.float64()),
        "high":   pa.array(np.round(hi.ravel()[mask], 2), type=pa.float64()),
        "low":    pa.array(np.round(lo.ravel()[mask], 2), type=pa.float64()),
        "close":  pa.array(np.round(c.ravel()[mask], 2), type=pa.float64()),
        "volume": pa.array(np.round(vol.ravel()[mask], 0), type=pa.float64()),
        "vwap":   pa.array(np.round(vwap.ravel()[mask], 4), type=pa.float64()),
        "trades": pa.array(trades.ravel()[mask], type=pa.int64()),
    })
    pq.write_table(tbl, os.path.join(done_dir, symbols[i] + ".parquet"),
                   compression="snappy")
    return rows

generated_this_run = 0
for i in range(TICKER_START, TICKER_END):
    if symbols[i] in done:
        continue
    rows = gen_ticker(i)
    done.add(symbols[i])
    generated_this_run += 1

    now = time.time()
    if i % 100 == 0:
        elapsed = (now - run_start) / 60
        gb = shutil.disk_usage(WORK_ROOT).free / 1e9
        print(f"  [t={elapsed:5.1f}m] {len(done):>6,}/{TOTAL_TICKERS:,} "
              f"done (+{generated_this_run:,} this run)  free={gb:6.1f} GB", flush=True)

    if now > gen_deadline:
        print("TIME BUDGET REACHED - stopping generation (publishing next)")
        break
    if now > hard_deadline:
        print("HARD DEADLINE REACHED - stopping generation (publishing next)")
        break
    if shutil.disk_usage(WORK_ROOT).free / 1e9 < 5.0:
        print("LOW DISK - stopping generation (publishing next)")
        break

gen_minutes = (time.time() - run_start) / 60
print(f"Generated {generated_this_run:,} tickers this run in {gen_minutes:.1f} min "
      f"({generated_this_run / max(gen_minutes, 0.1):,.0f} tickers/min)")


In [ ]:

# ============================================================================
# Write metadata files: tickers, types, manifest (only for DONE tickers)
# ============================================================================
summary_dir = os.path.join(LOCAL_DATA, "summary", "tickers")
types_dir   = os.path.join(LOCAL_DATA, "types")
os.makedirs(summary_dir, exist_ok=True)
os.makedirs(types_dir, exist_ok=True)

done_sorted = sorted(done)

# --- summary/tickers/tickers.parquet (mimics Massive list_tickers) ---------
type_lookup = {s: t for s, t in zip(symbols, ticker_types)}
tickers_df = pd.DataFrame({
    "ticker":            done_sorted,
    "type":              [type_lookup[t] for t in done_sorted],
    "name":              ["Synthetic " + t for t in done_sorted],
    "market":            "stocks",
    "locale":            "us",
    "currency_name":     "usd",
    "primary_exchange":  "SYNTH",
    "active":            True,
})
tickers_df.to_parquet(os.path.join(summary_dir, "tickers.parquet"))
print(f"tickers.parquet: {len(tickers_df):,} rows")

# --- types/ticker_types.parquet (asset-class lookup, mimics Massive) -------
types_df = pd.DataFrame({
    "asset_class": ["stocks"] * 12,
    "code":        ["CS", "PFD", "WARRANT", "RIGHT", "BOND", "ETF",
                    "ETN", "ETV", "ETS", "SP", "ADRC", "UNIT"],
    "description": ["Common Stock", "Preferred Stock", "Warrant", "Rights",
                    "Corporate Bond", "Exchange Traded Fund", "Exchange Traded Note",
                    "Exchange Traded Vehicle", "Exchange Traded Something",
                    "Structured Product", "ADR Common", "Unit"],
    "locale":      ["us"] * 12,
})
types_df.to_parquet(os.path.join(types_dir, "ticker_types.parquet"))
print(f"ticker_types.parquet: {len(types_df):,} rows")

# --- manifest.json (resume state) ------------------------------------------
manifest["params"] = {
    "total_tickers": TOTAL_TICKERS, "seed": SEED, "n_sectors": N_SECTORS,
    "start_date": START_DATE, "end_date": END_DATE,
    "bars_per_day": BARS_PER_DAY, "minute_step": MINUTE_STEP,
    "dataset": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
}
manifest["run_log"].append({
    "run": run_number,
    "ts": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "generated_this_run": generated_this_run,
    "tickers_done": len(done_sorted),
    "total_tickers": TOTAL_TICKERS,
    "generation_minutes": round(gen_minutes, 1),
})
manifest["done"] = done_sorted
with open(os.path.join(LOCAL_DATA, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)
print(f"manifest.json: {len(done_sorted):,} tickers done, run log entries: {len(manifest['run_log'])}")


In [ ]:

# ============================================================================
# Sanity checks
# ============================================================================
files = glob.glob(os.path.join(done_dir, "*.parquet"))
total_rows = 0
for f in files:
    total_rows += pq.ParquetFile(f).metadata.num_rows
gb = sum(os.path.getsize(f) for f in files) / 1e9

print(f"minute files : {len(files):,}")
print(f"total rows   : {total_rows:,}")
print(f"total size   : {gb:.1f} GB")
print(f"rows/ticker  : {total_rows // max(len(files),1):,}")

# per-ticker coverage check (defective tickers should be < full count)
cal_days = (pd.Timestamp(END_DATE) - pd.Timestamp(START_DATE)).days
cov = []
for f in files[:200]:
    d = pq.read_table(f).column("date").to_pandas()
    cov.append((d.max() - d.min()) / (cal_days * 86400 * 1000))
print(f"sample calendar span (first 200): {np.mean(cov)*100:.1f}% "
      f"(~100% = full window incl. weekends)")

# show one row of a random ticker
sample = pq.read_table(files[0]).to_pandas()
print("\nSample rows:")
print(sample.head(3).to_string())


In [ ]:

# ============================================================================
# Publish a new dataset version (create on first run, version thereafter)
# ============================================================================
meta = {
    "title": DATASET_TITLE,
    "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "isPrivate": DATASET_PRIVATE,
    "licenses": [{"name": "CC0-1.0"}],
}
with open(os.path.join(UPLOAD_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

msg = f"run {run_number}: {len(done_sorted):,}/{TOTAL_TICKERS:,} tickers"

print("=" * 70)
print(f"PUBLISHING: {len(done_sorted):,}/{TOTAL_TICKERS:,} tickers (run #{run_number})")
print(f"  staging dir : {UPLOAD_DIR}")
print(f"  dataset     : {KAGGLE_USERNAME}/{DATASET_SLUG} (public={not DATASET_PRIVATE})")
print(f"  message     : {msg}")
print("=" * 70)

cmd = (f"kaggle datasets version -p {UPLOAD_DIR} --dir-mode zip "
       f"-m \"{msg}\"")
rc = os.system(cmd)
if rc != 0:
    print("  `datasets version` failed (dataset may not exist yet) -> trying `create`")
    rc = os.system(f"kaggle datasets create -p {UPLOAD_DIR}")

# The version is created asynchronously; confirm the dataset has content.
published = (rc == 0)
if published:
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        from kagglesdk.datasets.types.dataset_api_service import ApiGetDatasetRequest
        _api = KaggleApi(); _api.authenticate()
        _client = _api.build_kaggle_client().datasets.dataset_api_client
        _req = ApiGetDatasetRequest()
        _req.dataset_slug = DATASET_SLUG
        _req.owner_slug = KAGGLE_USERNAME
        _d = _client.get_dataset(_req)
        _bytes = _d.total_bytes
        published = _bytes > 0
        print(f"  dataset total_bytes so far: {_bytes:,} (version still processing if < staging size)")
    except Exception:
        published = True   # CLI accepted the upload; processing is async

print()
if published:
    print("DONE - dataset version accepted for processing.")
else:
    print("PUBLISH FAILED - check the error above; the staging area is intact, "
          "so a re-run will resume from where it stopped.")

# --- summary + next-step hint ----------------------------------------------
shard_total = TICKER_END - TICKER_START
pct = len(done_sorted) / shard_total * 100
print("\n" + "=" * 70)
print(f"  shard {DATASET_SLUG}: {len(done_sorted):,}/{shard_total:,} tickers generated ({pct:.1f}%)")
print(f"  global: SGT{TICKER_START:05d} .. SGT{TICKER_END-1:05d}")
print(f"  dataset: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")
if len(done_sorted) < TOTAL_TICKERS:
    print("  NEXT STEP: re-push this kernel to keep building "
          "(it resumes from the manifest).")
else:
    print("  COMPLETE: the synthetic bucket is fully built. Run 02-01/03-01/04-01 "
          "against this dataset to demo the pipeline.")
print("=" * 70)
